In [ ]:
import pandas as pd

# 1. Carregar o arquivo
df = pd.read_csv('repasse-s.csv', sep=';', encoding='latin1')

# 2. Exibir os nomes de todas as colunas
print("--- Nomes das Colunas ---")
print(df.columns.tolist())

# 3. Mostrar as 5 primeiras linhas da tabela
print("\n--- Amostra dos Dados ---")
df.head()

--- Nomes das Colunas ---
['ï»¿MÃªs / Ano de ReferÃªncia', 'Entidade', 'UC/CNPJ', 'Total Repassado']

--- Amostra dos Dados ---


,ï»¿MÃªs / Ano de ReferÃªncia,Entidade,UC/CNPJ,Total Repassado
0,jan/15,FNDE,153173 / 15253,"2.604.077.590,00"
1,jan/15,INCRA,373001 / 37201,"181.456.016,00"
2,jan/15,SENAI,33.564.543/0001-90,"219.119.330,00"
3,jan/15,SESI,33.641.358/0001-52,"313.370.139,00"
4,jan/15,SENAC,33.469.172/0001-68,"334.005.249,00"


In [ ]:
import pandas as pd

# 1. Carregar o arquivo
df = pd.read_csv('repasse-s.csv', sep=';', encoding='latin1')

# 2. Dropar a coluna UC/CNPJ
df = df.drop(columns=['UC/CNPJ'])

# 3. Exibir os nomes atualizados das colunas
print("--- Colunas Restantes ---")
print(df.columns.tolist())

# 4. Mostrar as 5 primeiras linhas da tabela
print("\n--- Amostra dos Dados ---")
df.head()

--- Colunas Restantes ---
['ï»¿MÃªs / Ano de ReferÃªncia', 'Entidade', 'Total Repassado']

--- Amostra dos Dados ---


,ï»¿MÃªs / Ano de ReferÃªncia,Entidade,Total Repassado
0,jan/15,FNDE,"2.604.077.590,00"
1,jan/15,INCRA,"181.456.016,00"
2,jan/15,SENAI,"219.119.330,00"
3,jan/15,SESI,"313.370.139,00"
4,jan/15,SENAC,"334.005.249,00"


In [ ]:
# 1. Renomear as 3 colunas para nomes limpos
df.columns = ['mes_ano', 'entidade', 'total_repassado']

# 2. Mapeamento dos meses (nome por extenso e número)
meses_nome = {'jan': 'Janeiro', 'fev': 'Fevereiro', 'mar': 'Março', 'abr': 'Abril',
              'mai': 'Maio', 'jun': 'Junho', 'jul': 'Julho', 'ago': 'Agosto',
              'set': 'Setembro', 'out': 'Outubro', 'nov': 'Novembro', 'dez': 'Dezembro'}

meses_num = {'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
             'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'}

# 3. Separar mês e ano a partir da coluna 'mes_ano' (ex: jan/15 -> mes: jan, ano: 2015)
df['mes_txt'] = df['mes_ano'].apply(lambda x: str(x).strip().split('/')[0].lower())
df['ano'] = df['mes_ano'].apply(lambda x: int('20' + str(x).strip().split('/')[1]))

# 4. Criar as novas colunas
df['mes_nome'] = df['mes_txt'].map(meses_nome)
df['mes_num'] = df['mes_txt'].map(meses_num)

# 5. Criar coluna de Data oficial no padrão internacional (AAAA-MM-DD)
df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes_num'] + '-01')

# 6. Tratar o valor monetário (converter para float)
df['valor_repassado'] = (
    df['total_repassado']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 7. Padronizar nome da entidade
df['entidade'] = df['entidade'].astype(str).str.strip().replace({'APEX-BR': 'APEX'})

# 8. Filtrar o período de 2020 a 2026 e organizar as colunas
df_final = df[df['ano'].between(2020, 2026)][
    ['data', 'ano', 'mes_num', 'mes_nome', 'entidade', 'valor_repassado']
]

# Exibir a prévia das primeiras linhas da tabela limpa
df_final.head()

IndexError: list index out of range

In [ ]:
import pandas as pd

# 1. Carregar o arquivo
df = pd.read_csv('repasse-s.csv', sep=';', encoding='latin1')

# 2. Dropar a coluna UC/CNPJ se ela existir
if 'UC/CNPJ' in df.columns:
    df = df.drop(columns=['UC/CNPJ'])

# 3. Renomear IMEDIATAMENTE as 3 colunas para evitar erros de acentuação/encoding
df.columns = ['mes_ano', 'entidade', 'total_repassado']

# 4. Remover linhas vazias e garantir que só temos textos válidos no formato "mes/ano" (ex: jan/15)
df = df.dropna(subset=['mes_ano']).copy()
df['mes_ano'] = df['mes_ano'].astype(str).str.strip()
df = df[df['mes_ano'].str.contains('/', na=False)].copy()

# 5. Mapeamento dos meses
meses_nome = {'jan': 'Janeiro', 'fev': 'Fevereiro', 'mar': 'Março', 'abr': 'Abril',
              'mai': 'Maio', 'jun': 'Junho', 'jul': 'Julho', 'ago': 'Agosto',
              'set': 'Setembro', 'out': 'Outubro', 'nov': 'Novembro', 'dez': 'Dezembro'}

meses_num = {'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
             'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'}

# 6. Separar Mês e Ano com segurança total
def extrair_mes(val):
    partes = val.split('/')
    return partes[0].lower() if len(partes) == 2 else None

def extrair_ano(val):
    partes = val.split('/')
    if len(partes) == 2 and partes[1].isdigit():
        return int('20' + partes[1])
    return None

df['mes_txt'] = df['mes_ano'].apply(extrair_mes)
df['ano'] = df['mes_ano'].apply(extrair_ano)

# Remover qualquer linha que não tenha conseguido extrair o ano/mês corretamente
df = df.dropna(subset=['ano', 'mes_txt']).copy()
df['ano'] = df['ano'].astype(int)

# 7. Criar colunas de mês e data oficial ISO
df['mes_nome'] = df['mes_txt'].map(meses_nome)
df['mes_num'] = df['mes_txt'].map(meses_num)
df['data'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes_num'] + '-01')

# 8. Tratar valor monetário (converter para float)
df['valor_repassado'] = (
    df['total_repassado']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 9. Padronizar nome da entidade
df['entidade'] = df['entidade'].astype(str).str.strip().replace({'APEX-BR': 'APEX'})

# 10. Filtrar de 2020 a 2026 e organizar colunas
df_final = df[df['ano'].between(2020, 2026)][
    ['data', 'ano', 'mes_num', 'mes_nome', 'entidade', 'valor_repassado']
]

# Exibir as 5 primeiras linhas do resultado final
df_final.head()

,data,ano,mes_num,mes_nome,entidade,valor_repassado
1020,2020-01-01,2020,01,Janeiro,FNDE,3.023695e+09
1021,2020-01-01,2020,01,Janeiro,INCRA,2.529641e+08
1022,2020-01-01,2020,01,Janeiro,SENAI,1.993244e+08
1023,2020-01-01,2020,01,Janeiro,SESI,2.869794e+08
1024,2020-01-01,2020,01,Janeiro,SENAC,4.240555e+08


In [ ]:
# Filtrar de 2020 a 2026 e organizar as colunas (SEM 'data')
df_final = df[df['ano'].between(2020, 2026)][
    ['ano', 'mes_num', 'mes_nome', 'entidade', 'valor_repassado']
]

# Salvar o arquivo limpo
df_final.to_csv('repasses_sistema_s_2020_2026_limpo.csv', index=False, encoding='utf-8-sig')

# Exibir as 5 primeiras linhas do resultado final
df_final.head()

,ano,mes_num,mes_nome,entidade,valor_repassado
1020,2020,01,Janeiro,FNDE,3.023695e+09
1021,2020,01,Janeiro,INCRA,2.529641e+08
1022,2020,01,Janeiro,SENAI,1.993244e+08
1023,2020,01,Janeiro,SESI,2.869794e+08
1024,2020,01,Janeiro,SENAC,4.240555e+08


In [ ]:
import pandas as pd
from google.colab import files

# 1. Carregar o arquivo
df = pd.read_csv('repasse-s.csv', sep=';', encoding='latin1')

# 2. Dropar a coluna UC/CNPJ se ela existir
if 'UC/CNPJ' in df.columns:
    df = df.drop(columns=['UC/CNPJ'])

# 3. Renomear colunas
df.columns = ['mes_ano', 'entidade', 'total_repassado']

# 4. Remover linhas vazias e filtrar valores válidos
df = df.dropna(subset=['mes_ano']).copy()
df['mes_ano'] = df['mes_ano'].astype(str).str.strip()
df = df[df['mes_ano'].str.contains('/', na=False)].copy()

# 5. Mapeamento dos meses
meses_nome = {'jan': 'Janeiro', 'fev': 'Fevereiro', 'mar': 'Março', 'abr': 'Abril',
              'mai': 'Maio', 'jun': 'Junho', 'jul': 'Julho', 'ago': 'Agosto',
              'set': 'Setembro', 'out': 'Outubro', 'nov': 'Novembro', 'dez': 'Dezembro'}

meses_num = {'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
             'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'}

# 6. Extrair Mês e Ano
def extrair_mes(val):
    partes = val.split('/')
    return partes[0].lower() if len(partes) == 2 else None

def extrair_ano(val):
    partes = val.split('/')
    if len(partes) == 2 and partes[1].isdigit():
        return int('20' + partes[1])
    return None

df['mes_txt'] = df['mes_ano'].apply(extrair_mes)
df['ano'] = df['mes_ano'].apply(extrair_ano)

# Limpar nulos de data
df = df.dropna(subset=['ano', 'mes_txt']).copy()
df['ano'] = df['ano'].astype(int)

# 7. Criar colunas de mês
df['mes_nome'] = df['mes_txt'].map(meses_nome)
df['mes_num'] = df['mes_txt'].map(meses_num)

# 8. Tratar valor monetário (converter para float)
df['valor_repassado'] = (
    df['total_repassado']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 9. Padronizar nome da entidade
df['entidade'] = df['entidade'].astype(str).str.strip().replace({'APEX-BR': 'APEX'})

# 10. Filtrar de 2020 a 2026 e organizar colunas (SEM a coluna 'data')
df_final = df[df['ano'].between(2020, 2026)][
    ['ano', 'mes_num', 'mes_nome', 'entidade', 'valor_repassado']
]

# 11. Salvar o arquivo limpo
nome_arquivo = 'repasses_sistema_s_2020_2026_limpo.csv'
df_final.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

# 12. Fazer o download automático para o seu computador
files.download(nome_arquivo)

# Exibir prévia na tela
df_final.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,ano,mes_num,mes_nome,entidade,valor_repassado
1020,2020,01,Janeiro,FNDE,3.023695e+09
1021,2020,01,Janeiro,INCRA,2.529641e+08
1022,2020,01,Janeiro,SENAI,1.993244e+08
1023,2020,01,Janeiro,SESI,2.869794e+08
1024,2020,01,Janeiro,SENAC,4.240555e+08
